# 03 - Fine-Tuning

Fine-tunes Mistral 7B Instruct v0.3 on our synthetic function-calling dataset using QLoRA. Trains the model to reliably select the correct tool and produce valid JSON arguments for 16 enterprise tool schemas.

**Goal**: Improve structured function-calling accuracy over the baseline established in Notebook 02.

In [0]:
%pip install -q transformers accelerate bitsandbytes peft trl datasets tqdm

In [0]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN"))

# Verify login
user_info = whoami()
print(f"Logged in as: {user_info['name']}")

In [0]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN"))

user_info = whoami()
print(f"Logged in as: {user_info['name']}")

import sys
import os
import pandas as pd

# Add project root to path
PROJECT_ROOT = "/Workspace/Users/alberto.lapedriza@kpmg.co.uk/mistral-7b-enterprise-function-calling"
sys.path.insert(0, PROJECT_ROOT)

from src.utils import load_jsonl, dataset_stats, spot_check
from src.training import (
    load_model_for_training,
    apply_lora,
    run_training,
    check_truncation,
    merge_system_into_user,
    DEFAULT_LORA_CONFIG,
    DEFAULT_TRAINING_ARGS,
    MAX_SEQ_LENGTH,
)

In [0]:
DATA_DIR = f"{PROJECT_ROOT}/data"
OUTPUT_DIR = f"{PROJECT_ROOT}/outputs/qlora-run-1"

train_data = load_jsonl(f"{DATA_DIR}/train.jsonl")
val_data = load_jsonl(f"{DATA_DIR}/val.jsonl")

print(f"Train: {len(train_data)} examples")
print(f"Val:   {len(val_data)} examples")
dataset_stats(train_data)

In [0]:
model, tokenizer = load_model_for_training()
model = apply_lora(model)

In [0]:
# Verify tokenization works on a sample after merging system into user
sample_messages = merge_system_into_user(train_data[0]["messages"])
formatted = tokenizer.apply_chat_template(sample_messages, tokenize=False)
print(f"Sample formatted length: {len(formatted)} chars")
print(f"Sample token count: {len(tokenizer.encode(formatted))}")
print(f"\n--- Formatted sample (first 500 chars) ---\n{formatted[:500]}")

In [0]:

# Check if any examples will be truncated at current MAX_SEQ_LENGTH.
# Review the output — cancel execution if truncation is unacceptable.
train_stats = check_truncation(train_data, tokenizer, MAX_SEQ_LENGTH, label="train")
val_stats = check_truncation(val_data, tokenizer, MAX_SEQ_LENGTH, label="val")
stats = pd.concat([train_stats, val_stats], ignore_index=True)
stats.head()

In [0]:
trainer, train_result = run_training(
    model=model,
    tokenizer=tokenizer,
    train_data=train_data,
    val_data=val_data,
    output_dir=OUTPUT_DIR,
)

In [0]:
# Log training metrics
metrics = train_result.metrics
print(f"Training loss:     {metrics['train_loss']:.4f}")
print(f"Training runtime:  {metrics['train_runtime']:.1f}s")
print(f"Samples/second:    {metrics['train_samples_per_second']:.2f}")
print(f"Steps:             {metrics['train_steps']}")

# Evaluate on validation set
eval_metrics = trainer.evaluate()
print(f"\nValidation loss:   {eval_metrics['eval_loss']:.4f}")

In [0]:
# Quick sanity check: generate on a training example to verify the adapter loaded
from src.inference import run_inference

model.eval()
test_messages = train_data[0]["messages"][:2]  # system + user only

prediction = run_inference(model, tokenizer, test_messages)
expected = train_data[0]["messages"][2]["content"]

print("USER:", test_messages[1]["content"][:200])
print(f"\nEXPECTED:\n{expected[:300]}")
print(f"\nPREDICTED:\n{prediction[:300]}")